In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import random
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [ ]:
data_1 = pd.read_csv("./data/bruna/lvedv/ukbb_all_no_exclusion_all_2_and_3_with_CMR_and_MR", delimiter='\t')
data_2 = pd.read_csv("./data/shriya/LV_strain_participant.tsv", delimiter='\t')

In [ ]:
data_1_subset = data_1[['id', "LVEF", "LVM (g)", "LVESV (mL)", "age (y)", "Sex" , "stroke_volume", "body surface area estimate (m2)", "Body mass index (BMI) | Instance 2", "Body fat percentage | Instance 2"]].copy()

cluster_data = pd.merge(data_1_subset, data_2[["eid", "p24181_i2", "p24174_i2"]], left_on='id', right_on='eid', how='inner')

In [ ]:
cluster_data.replace(-1000, np.nan, inplace=True)
cluster_data.dropna(inplace=True)

In [ ]:
cluster_data = cluster_data.drop(columns=['eid'])
cluster_data = cluster_data.rename(columns={'p24181_i2': 'LV_strain_longitudinal', 'p24174_i2': 'LV_strain_radial'})
cluster_data['Sex'] = cluster_data['Sex'].map({'Male': 0, 'Female': 1})

In [ ]:
# Define independent variables (predictors): sex, age, and body surface area
X = cluster_data[['Sex', 'age (y)', 'body surface area estimate (m2)']]
X_noBSA = cluster_data[['Sex', 'age (y)']]

# Add a constant (intercept) to the model
X = sm.add_constant(X)
X_noBSA = sm.add_constant(X_noBSA)

# Dependent variables to regress
dependent_vars = ['LVM (g)', 'LVESV (mL)', 'LV_strain_longitudinal', 'LV_strain_radial']

# Initialize an empty DataFrame to store the residuals
residuals_df = pd.DataFrame()
residuals_patientID = cluster_data[['id']]

# Regress each dependent variable by sex, age, and BSA and collect residuals
for var in dependent_vars:
    y = cluster_data[var]
    model = sm.OLS(y, X).fit()  # Ordinary Least Squares regression
    residuals_df[f'{var}_regressed'] = model.resid  # Store residuals in new columns
    residuals_patientID[f'{var}_regressed'] = model.resid  # Store residuals in new columns

y = cluster_data['LVEF']
model = sm.OLS(y, X_noBSA).fit()  # Ordinary Least Squares regression
residuals_df[f'LVEF_regressed'] = model.resid  # Store residuals in new columns
residuals_patientID[f'LVEF_regressed'] = model.resid  # Store residuals in new columns

In [ ]:
regressed_cluster_data = residuals_df

In [ ]:
from scipy.stats import pearsonr

# Calculate the Pearson correlation coefficient and the p-value
pearson_corr, p_value = pearsonr(cluster_data['LV_strain_longitudinal'], cluster_data['LV_strain_radial'])

# Display the results
print("Pearson correlation coefficient:", pearson_corr)
print("P-value:", p_value)

# Calculate the Pearson correlation coefficient and the p-value
pearson_corr, p_value = pearsonr(regressed_cluster_data['LV_strain_longitudinal_regressed'], regressed_cluster_data['LV_strain_radial_regressed'])

# Display the results
print("Pearson regressed_cluster_data correlation coefficient:", pearson_corr)
print("P-value:", p_value)

In [ ]:
# Create a scatter plot to visualize the relationship between the two regressed variables
plt.figure(figsize=(10, 6))
sns.scatterplot(x='LV_strain_longitudinal', y='LV_strain_radial', data=cluster_data)

# Add labels and title
plt.xlabel('LV Strain Longitudinal')
plt.ylabel('LV Strain Radial')
plt.title('Scatter Plot of Raw LV Strain Longitudinal vs Radial')

# Show the plot
plt.show()

In [ ]:
# Create a scatter plot to visualize the relationship between the two regressed variables
plt.figure(figsize=(10, 6))
sns.scatterplot(x='LV_strain_longitudinal_regressed', y='LV_strain_radial_regressed', data=regressed_cluster_data)

# Add labels and title
plt.xlabel('LV Strain Longitudinal (Regressed)')
plt.ylabel('LV Strain Radial (Regressed)')
plt.title('Scatter Plot of Regressed LV Strain Longitudinal vs Radial')

# Show the plot
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(regressed_cluster_data)

# Performing PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(scaled_data)

# Create a DataFrame for PCA results
pca_df = pd.DataFrame(data=pca_result, columns=['PC1', 'PC2', 'PC3'])

### K-Means Classification

In [ ]:
# Perform K-Means clustering
kmeans = KMeans(n_clusters=3, random_state=42)
pca_df['Cluster'] = kmeans.fit_predict(pca_df[['PC1', 'PC2', 'PC3']])
regressed_cluster_data['Cluster'] = kmeans.fit_predict(pca_df[['PC1', 'PC2', 'PC3']])

In [ ]:
# Visualize the clusters
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot
scatter = ax.scatter(pca_df['PC1'], pca_df['PC2'], pca_df['PC3'], c=pca_df['Cluster'], cmap='viridis')

# Add legend
legend1 = ax.legend(*scatter.legend_elements(), title="Clusters")
ax.add_artist(legend1)

# Label axes
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
ax.set_title('K-Means Clustering with 3 Principal Components')

# Show plot
plt.show()

In [ ]:
# Define the phenotypes to plot
phenotypes = ['LVEF_regressed', 'LVM (g)_regressed', 'LVESV (mL)_regressed', 'LV_strain_longitudinal_regressed', 'LV_strain_radial_regressed']

# Create box plots for each phenotype
plt.figure(figsize=(16, 10))

for i, phenotype in enumerate(phenotypes, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(x='Cluster', y=phenotype, data=regressed_cluster_data)
    plt.title(f'Box Plot of {phenotype} by Cluster')
    plt.xlabel('Cluster')
    plt.ylabel(phenotype)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate and print summary statistics
summary_stats = regressed_cluster_data.groupby('Cluster')[phenotypes].agg(['mean', 'median', 'std', 
                                                                 lambda x: len(x)])
summary_stats.columns = ['_'.join(map(str, col)).strip() for col in summary_stats.columns.values]

# Print summary statistics
print(summary_stats)

In [ ]:
results = {}
clusters = [0, 1, 2]

for phenotype in phenotypes:
    results[phenotype] = {}
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            cluster1 = regressed_cluster_data[regressed_cluster_data['Cluster'] == clusters[i]][phenotype]
            cluster2 = regressed_cluster_data[regressed_cluster_data['Cluster'] == clusters[j]][phenotype]
            
            # Perform the t-test
            t_stat, p_value = stats.ttest_ind(cluster1, cluster2, equal_var=False)  # Use equal_var=False for Welch's t-test
            
            results[phenotype][f'Cluster {clusters[i]} vs Cluster {clusters[j]}'] = {
                't-statistic': t_stat,
                'p-value': p_value,
                'meanA': len(cluster1),
                'meanB': len(cluster2)
            }

# Print results
for phenotype, comparisons in results.items():
    print(f"\nT-test results for {phenotype}:")
    for comparison, stats in comparisons.items():
        print(f"{comparison}: t-statistic = {stats['t-statistic']:.4f}, p-value = {stats['p-value']:.4f}, N_A = {stats['meanA']:.4f}, N_B = {stats['meanB']:.4f}")

In [ ]:
# Concatenate the two DataFrames
concatenated_data = pd.concat([residuals_patientID, regressed_cluster_data], axis=1)
identified_clusters = concatenated_data[["id", "Cluster"]]
# Display the resulting DataFrame
identified_clusters

In [ ]:
# Save all IDs of clusters to a txt file
cluster_sets = identified_clusters.groupby('Cluster')['id'].apply(set).reset_index()
cluster_sets.columns = ['Cluster', 'IDs']

mapping_str = '\n'.join(f"Cluster {row['Cluster']}: {', '.join(map(str, row['IDs']))}" for _, row in cluster_sets.iterrows())

In [ ]:
file_path = './data/shriya/t1_myocardium_gwas_sensitivity_analysis/identified_clusters_IDs.txt'

# Save the mapping to a text file
with open(file_path, 'w') as file:
    file.write(mapping_str)

In [ ]:
import re

In [ ]:
# Parse disease_patient_IDs.txt to create a dictionary of diseases and IDs
disease_dict = {}

# Expected categories
expected_categories = ['HCM', 'DCM', 'Valvular', 'RestrictiveCM', 'Amyloidosis', 'Ischemic', 'NonIschemic', 'Normal']

# Open and read the disease_patient_IDs.txt file
with open("./data/shriya/SHMOLLI-output-unet-myocardium/disease_patient_IDs.txt", "r") as file:
    for line in file:
        # Match the line that contains the disease name and patient IDs
        match = re.match(r"(\w+[\s\w]*)\s+IDs:\s+(.+)", line)  # Adjust regex to capture multi-word disease names
        if match:
            disease = match.group(1).strip()  # Get disease name and strip whitespace
            ids = list(map(int, match.group(2).split(", ")))  # Get list of patient IDs
            # Store in dictionary if the disease is in the expected categories
            if disease in expected_categories:
                disease_dict[disease] = ids

In [ ]:
# Add disease status columns to concatenated_data
for disease, ids in disease_dict.items():
    concatenated_data[f"{disease}"] = concatenated_data["id"].isin(ids).astype(int)

# Generate and display a separate bar plot for each disease
for disease in disease_dict.keys():
    # Filter data to count the number of patients per cluster for the disease
    disease_counts = concatenated_data.groupby("Cluster")[f"{disease}"].sum()
    
    # Plotting
    plt.figure(figsize=(8, 6))
    disease_counts.plot(kind="bar", color="skyblue")
    plt.title(f"Number of Patients with {disease} Across Clusters")
    plt.xlabel("Cluster")
    plt.ylabel("Number of Patients")
    plt.xticks(rotation=0)
    plt.show()

In [ ]:
patient_mapping = pd.read_table("./data/shriya/ukb22282_24983_mapping.tsv", header = None)

# Function to get the value from column 0 based on the ID in column 1
def get_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[1].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[1] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return mapping_df.loc[row_index, 0]
    else:
        return None  # Return None if the ID is not found

# Function to get the value from column 0 based on the ID in column 1
def reverse_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[0].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[0] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return int(mapping_df.loc[row_index, 1])
    else:
        return None  # Return None if the ID is not found

In [ ]:
# Load the main dataset
input_file = './data/shriya/SHMOLLI-output-unet-myocardium/combined_T1_phenotypes_imputed.no_outliers.residuals.qnorm'
data = pd.read_csv(input_file, sep='\t')


In [ ]:
# Filter data and save files for each cluster
for i in range(len(cluster_sets)):
    iids = cluster_sets["IDs"][i]
    mapped_iids = list(map(lambda iid: reverse_mapped_ID(iid, patient_mapping), iids))
    
    # Filter the data based on the current cluster's IIDs
    cluster_data = data[data['FID'].isin(mapped_iids)]

    print(len(cluster_data))
    
    # Define the output file name for the current cluster
    output_file = f'./data/shriya/SHMOLLI-output-unet-myocardium/cluster_{i}_T1_phenotypes_imputed.no_outliers.residuals.qnorm'
    
    # Save the filtered data to a new file
    cluster_data.to_csv(output_file, sep='\t', index=False)
    print(f'File saved for Cluster {i}: {output_file}')